## Anatomy of an Ingress resource

A minimal but realistic Ingress:

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: web
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /   # controller-specific
spec:
  ingressClassName: nginx
  tls:
    - hosts: [www.example.com]
      secretName: web-tls                            # kubernetes.io/tls (notebook 05)
  rules:
    - host: www.example.com
      http:
        paths:
          - path: /api
            pathType: Prefix
            backend: { service: { name: api, port: { number: 80 } } }
          - path: /
            pathType: Prefix
            backend: { service: { name: web, port: { number: 80 } } }
```

Field by field:

- **`spec.tls`** — TLS termination. Binds hostnames to a `kubernetes.io/tls` Secret; the controller serves HTTPS for those hosts.
- **`spec.rules`** — host-based routing; each rule matches a `Host:` header.
- **`http.paths`** — path routing. `path` + **`pathType`** (`Prefix` matches `/api`, `/api/`, `/api/foo`; `Exact` only `/api`) + `backend.service`.
- **Annotations** — controller-specific magic: rewrite targets, HTTP→HTTPS redirect, timeouts, basic auth. `ingress-nginx` uses `nginx.ingress.kubernetes.io/...`; ALB uses `alb...`.

**The TLS Secret** must be `type: kubernetes.io/tls` with `tls.crt` + `tls.key`. Get one via `kubectl create secret tls`, **cert-manager** (auto Let's Encrypt — the prod default), or a controller's cloud-cert integration. Unmatched requests hit a **default backend** (usually a 404 pod). On our map this is the **Ingress** chip routing down onto the **Service** chips behind it — the concrete wiring of host+path → Service.